In [2]:
# ============================================================
# CareerIQ
# Salary Prediction - Step 3
# Skill Cleaning + Domain + Role Creation
# ============================================================

import os
import re
import pandas as pd
import numpy as np


# ============================================================
# 1. PATHS
# ============================================================

CLEANED_DATASET_PATH = (
    "../data/salary_prediction/processed/"
    "salary_prediction_cleaned.xlsx"
)

OUTPUT_DATASET_PATH = (
    "../data/salary_prediction/processed/"
    "salary_prediction_features.xlsx"
)


# ============================================================
# 2. LOAD CLEANED DATASET
# ============================================================

df = pd.read_excel(CLEANED_DATASET_PATH)

print("=" * 60)
print("CLEANED DATASET LOADED")
print("=" * 60)

print(f"Rows    : {len(df):,}")
print(f"Columns : {len(df.columns)}")


# ============================================================
# 3. CHECK REQUIRED COLUMNS
# ============================================================

required_columns = [
    "title",
    "tagsandskills",
    "jobdescription",
    "experience_years",
    "salary_target"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )


# ============================================================
# 4. TEXT CLEANING FUNCTION
# ============================================================

def clean_text(value):
    """
    Clean text while preserving useful technical terms.
    """

    if pd.isna(value):
        return ""

    value = str(value).lower()

    # Replace HTML tags
    value = re.sub(r"<[^>]+>", " ", value)

    # Replace common separators
    value = re.sub(r"[/|,;]+", " ", value)

    # Keep letters, numbers, #, +, . and -
    value = re.sub(
        r"[^a-z0-9+#.\- ]+",
        " ",
        value
    )

    # Remove extra spaces
    value = re.sub(r"\s+", " ", value).strip()

    return value


# ============================================================
# 5. CLEAN JOB TITLE
# ============================================================

df["title_clean"] = (
    df["title"]
    .fillna("")
    .astype(str)
    .apply(clean_text)
)


# ============================================================
# 6. CLEAN REQUIRED SKILLS
# ============================================================

df["skills_clean"] = (
    df["tagsandskills"]
    .fillna("")
    .astype(str)
    .apply(clean_text)
)


# ============================================================
# 7. CLEAN JOB DESCRIPTION
# ============================================================

df["description_clean"] = (
    df["jobdescription"]
    .fillna("")
    .astype(str)
    .apply(clean_text)
)


# ============================================================
# 8. COMBINE INFORMATION FOR DOMAIN DETECTION
# ============================================================

df["domain_text"] = (
    df["title_clean"]
    + " "
    + df["skills_clean"]
    + " "
    + df["description_clean"]
)


# ============================================================
# 9. DOMAIN KEYWORDS
# ============================================================

DOMAIN_KEYWORDS = {

    "AI / Machine Learning": [
        "machine learning",
        "deep learning",
        "artificial intelligence",
        "artificial intelligence",
        "ai",
        "ml",
        "tensorflow",
        "pytorch",
        "keras",
        "scikit learn",
        "sklearn",
        "nlp",
        "natural language processing",
        "computer vision",
        "opencv",
        "generative ai",
        "genai",
        "llm",
        "large language model",
        "langchain",
        "rag",
        "neural network",
        "cnn",
        "rnn",
        "transformer",
        "xgboost"
    ],

    "Data Science": [
        "data science",
        "data scientist",
        "statistics",
        "statistical analysis",
        "pandas",
        "numpy",
        "matplotlib",
        "seaborn",
        "scipy",
        "jupyter",
        "data analysis",
        "predictive modeling",
        "exploratory data analysis",
        "eda"
    ],

    "Data Engineering": [
        "data engineer",
        "data engineering",
        "etl",
        "elt",
        "data pipeline",
        "apache spark",
        "spark",
        "hadoop",
        "kafka",
        "airflow",
        "databricks",
        "data warehouse",
        "data lake",
        "big data"
    ],

    "Software Development": [
        "software developer",
        "software engineer",
        "software development",
        "application developer",
        "backend developer",
        "backend engineer",
        "java developer",
        "python developer",
        "c++ developer",
        "c# developer",
        "spring boot",
        "spring",
        "node.js",
        "nodejs",
        "express",
        "api development",
        "rest api",
        "microservices"
    ],

    "Web Development": [
        "web developer",
        "web development",
        "frontend developer",
        "front end developer",
        "full stack developer",
        "fullstack developer",
        "react",
        "angular",
        "vue",
        "javascript",
        "typescript",
        "html",
        "css",
        "next.js",
        "nextjs"
    ],

    "Cloud / DevOps": [
        "devops",
        "devops engineer",
        "cloud engineer",
        "cloud computing",
        "aws",
        "amazon web services",
        "azure",
        "microsoft azure",
        "google cloud",
        "gcp",
        "docker",
        "kubernetes",
        "jenkins",
        "terraform",
        "ci cd",
        "continuous integration",
        "continuous deployment"
    ],

    "Cybersecurity": [
        "cyber security",
        "cybersecurity",
        "security engineer",
        "security analyst",
        "information security",
        "network security",
        "ethical hacking",
        "penetration testing",
        "penetration tester",
        "soc analyst",
        "siem",
        "vulnerability assessment",
        "application security"
    ],

    "Mobile Development": [
        "android developer",
        "android development",
        "ios developer",
        "ios development",
        "mobile developer",
        "mobile application",
        "flutter",
        "react native",
        "swift",
        "kotlin",
        "android studio"
    ],

    "QA / Testing": [
        "qa engineer",
        "qa tester",
        "quality assurance",
        "software testing",
        "manual testing",
        "automation testing",
        "selenium",
        "test automation",
        "quality analyst"
    ],

    "Product / Business Analytics": [
        "business analyst",
        "business analytics",
        "product analyst",
        "product analytics",
        "business intelligence",
        "bi analyst",
        "power bi",
        "tableau",
        "looker",
        "requirements analysis",
        "data visualization"
    ],

    "UI / UX": [
        "ui designer",
        "ux designer",
        "ui ux",
        "user experience",
        "user interface",
        "figma",
        "wireframe",
        "prototyping",
        "product designer"
    ]
}


# ============================================================
# 10. DOMAIN DETECTION FUNCTION
# ============================================================

def detect_domain(text):
    """
    Detect the most relevant technology domain
    using title + skills + job description.
    """

    text = str(text).lower()

    scores = {}

    for domain, keywords in DOMAIN_KEYWORDS.items():

        score = 0

        for keyword in keywords:

            # Give slightly higher importance to title matches
            if keyword in text:
                score += 1

        scores[domain] = score

    if not scores:
        return "Other"

    best_domain = max(
        scores,
        key=scores.get
    )

    if scores[best_domain] == 0:
        return "Other"

    return best_domain


# ============================================================
# 11. CREATE DOMAIN COLUMN
# ============================================================

print("\nDetecting job domains...")

df["domain"] = (
    df["domain_text"]
    .apply(detect_domain)
)


# ============================================================
# 12. ROLE CLEANING
# ============================================================

def clean_role(title):
    """
    Convert job titles into cleaner role names.
    """

    title = str(title).lower().strip()

    if not title:
        return "Other"

    # Remove common seniority terms
    title = re.sub(
        r"\b(senior|sr|junior|jr|lead|principal|"
        r"associate|trainee|intern|internship|"
        r"fresher|entry level|entry-level)\b",
        " ",
        title
    )

    # Remove experience numbers
    title = re.sub(
        r"\b\d+\+?\s*(years?|yrs?)\b",
        " ",
        title
    )

    # Remove extra spaces
    title = re.sub(
        r"\s+",
        " ",
        title
    ).strip()

    return title.title() if title else "Other"


# ============================================================
# 13. CREATE ROLE COLUMN
# ============================================================

df["role"] = (
    df["title_clean"]
    .apply(clean_role)
)


# ============================================================
# 14. CLEAN SKILL LIST
# ============================================================

def normalize_skills(skill_text):
    """
    Convert skill text into a standardized
    comma-separated skill list.
    """

    if not skill_text:
        return ""

    text = str(skill_text).lower()

    # Standardize common technology names
    replacements = {
        "node js": "node.js",
        "nodejs": "node.js",
        "react js": "react",
        "reactjs": "react",
        "next js": "next.js",
        "nextjs": "next.js",
        "angular js": "angular",
        "angularjs": "angular",
        "vue js": "vue",
        "vuejs": "vue",
        "scikit learn": "scikit-learn",
        "sklearn": "scikit-learn",
        "powerbi": "power bi",
        "ms excel": "excel",
        "microsoft excel": "excel",
        "amazon web services": "aws",
        "microsoft azure": "azure",
        "google cloud platform": "gcp",
        "artificial intelligence": "ai",
        "machine learning": "machine learning"
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    # Split on common separators
    skills = re.split(
        r"[,;|/\n]+",
        text
    )

    cleaned_skills = []

    for skill in skills:

        skill = skill.strip()

        if not skill:
            continue

        # Remove unnecessary punctuation
        skill = re.sub(
            r"\s+",
            " ",
            skill
        ).strip()

        if len(skill) < 2:
            continue

        if skill not in cleaned_skills:
            cleaned_skills.append(skill)

    return ", ".join(cleaned_skills)


# ============================================================
# 15. CREATE NORMALIZED SKILLS COLUMN
# ============================================================

print("Normalizing skills...")

df["required_skills"] = (
    df["tagsandskills"]
    .fillna("")
    .apply(normalize_skills)
)


# ============================================================
# 16. REMOVE TEMPORARY COLUMNS
# ============================================================

temporary_columns = [
    "domain_text"
]

df.drop(
    columns=[
        col
        for col in temporary_columns
        if col in df.columns
    ],
    inplace=True
)


# ============================================================
# 17. DOMAIN DISTRIBUTION
# ============================================================

print("\n" + "=" * 60)
print("DOMAIN DISTRIBUTION")
print("=" * 60)

domain_counts = (
    df["domain"]
    .value_counts()
)

print(domain_counts)


# ============================================================
# 18. TOP ROLES
# ============================================================

print("\n" + "=" * 60)
print("TOP 30 ROLES")
print("=" * 60)

role_counts = (
    df["role"]
    .value_counts()
    .head(30)
)

print(role_counts)


# ============================================================
# 19. CHECK SKILLS
# ============================================================

print("\n" + "=" * 60)
print("SKILL EXAMPLES")
print("=" * 60)

print(
    df[
        [
            "title",
            "required_skills",
            "domain",
            "role",
            "salary_target"
        ]
    ].head(10).to_string(index=False)
)


# ============================================================
# 20. FINAL DATASET INFORMATION
# ============================================================

print("\n" + "=" * 60)
print("FEATURE DATASET")
print("=" * 60)

print(f"Rows    : {len(df):,}")
print(f"Columns : {len(df.columns)}")

print("\nImportant columns:")

important_columns = [
    "title",
    "title_clean",
    "required_skills",
    "experience_years",
    "domain",
    "role",
    "salary_target"
]

for column in important_columns:

    if column in df.columns:
        print(f"✓ {column}")


# ============================================================
# 21. SAVE FEATURE DATASET
# ============================================================

os.makedirs(
    os.path.dirname(OUTPUT_DATASET_PATH),
    exist_ok=True
)

df.to_excel(
    OUTPUT_DATASET_PATH,
    index=False
)


# ============================================================
# 22. COMPLETION MESSAGE
# ============================================================

print("\n" + "=" * 60)
print("STEP 3 COMPLETED")
print("=" * 60)

print(
    f"Feature dataset saved to:\n"
    f"{OUTPUT_DATASET_PATH}"
)

CLEANED DATASET LOADED
Rows    : 32,505
Columns : 19

Detecting job domains...
Normalizing skills...

DOMAIN DISTRIBUTION
domain
Other                           14448
AI / Machine Learning           14072
Cloud / DevOps                    874
Web Development                   674
Software Development              666
Data Engineering                  533
QA / Testing                      374
Data Science                      333
Product / Business Analytics      208
Cybersecurity                     149
UI / UX                            98
Mobile Development                 76
Name: count, dtype: int64

TOP 30 ROLES
role
Sales Executive                          492
Business Development Executive           328
Accountant                               238
Business Development Manager             231
Sales Manager                            215
Customer Support Executive               179
Account Executive                        165
Java Full Stack Developer                157
Data Engin

In [3]:
# ============================================================
# CareerIQ
# Salary Prediction - Step 4
# ML Feature Preparation & Validation
# ============================================================

import os
import re
import numpy as np
import pandas as pd


# ============================================================
# 1. PATHS
# ============================================================

FEATURE_DATASET_PATH = (
    "../data/salary_prediction/processed/"
    "salary_prediction_features.xlsx"
)

OUTPUT_DATASET_PATH = (
    "../data/salary_prediction/processed/"
    "salary_prediction_ml_ready.xlsx"
)


# ============================================================
# 2. LOAD FEATURE DATASET
# ============================================================

df = pd.read_excel(FEATURE_DATASET_PATH)

print("=" * 60)
print("FEATURE DATASET LOADED")
print("=" * 60)

print(f"Rows    : {len(df):,}")
print(f"Columns : {len(df.columns)}")


# ============================================================
# 3. REQUIRED COLUMNS
# ============================================================

required_columns = [
    "title",
    "required_skills",
    "experience_years",
    "domain",
    "role",
    "salary_target"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )


# ============================================================
# 4. BASIC DATA VALIDATION
# ============================================================

print("\n" + "=" * 60)
print("DATA TYPES")
print("=" * 60)

print(
    df[
        required_columns
    ].dtypes
)


# ============================================================
# 5. CHECK MISSING VALUES
# ============================================================

print("\n" + "=" * 60)
print("MISSING VALUES")
print("=" * 60)

missing_values = (
    df[required_columns]
    .isnull()
    .sum()
    .sort_values(ascending=False)
)

print(missing_values)


# ============================================================
# 6. CLEAN REQUIRED FIELDS
# ============================================================

df["title"] = (
    df["title"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
)

df["required_skills"] = (
    df["required_skills"]
    .fillna("")
    .astype(str)
    .str.strip()
)

df["domain"] = (
    df["domain"]
    .fillna("Other")
    .astype(str)
    .str.strip()
)

df["role"] = (
    df["role"]
    .fillna("Other")
    .astype(str)
    .str.strip()
)


# ============================================================
# 7. EXPERIENCE CLEANING
# ============================================================

df["experience_years"] = pd.to_numeric(
    df["experience_years"],
    errors="coerce"
)

# Missing experience → 0
df["experience_years"] = (
    df["experience_years"]
    .fillna(0)
)

# Negative experience is invalid
df.loc[
    df["experience_years"] < 0,
    "experience_years"
] = 0


# ============================================================
# 8. SALARY TARGET CLEANING
# ============================================================

df["salary_target"] = pd.to_numeric(
    df["salary_target"],
    errors="coerce"
)

# Remove records without valid salary target
df = df[
    df["salary_target"].notna()
].copy()

# Remove zero / negative salaries
df = df[
    df["salary_target"] > 0
].copy()


# ============================================================
# 9. DOMAIN DISTRIBUTION
# ============================================================

print("\n" + "=" * 60)
print("DOMAIN DISTRIBUTION")
print("=" * 60)

domain_distribution = (
    df["domain"]
    .value_counts()
)

print(domain_distribution)


# ============================================================
# 10. DOMAIN PERCENTAGE
# ============================================================

print("\n" + "=" * 60)
print("DOMAIN PERCENTAGE")
print("=" * 60)

domain_percentage = (
    df["domain"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print(domain_percentage)


# ============================================================
# 11. ROLE DISTRIBUTION
# ============================================================

print("\n" + "=" * 60)
print("UNIQUE ROLES")
print("=" * 60)

print(
    f"Unique roles: {df['role'].nunique():,}"
)


print("\nTop 30 roles:")

print(
    df["role"]
    .value_counts()
    .head(30)
)


# ============================================================
# 12. RARE ROLE ANALYSIS
# ============================================================

role_counts = (
    df["role"]
    .value_counts()
)

print("\n" + "=" * 60)
print("RARE ROLE ANALYSIS")
print("=" * 60)

print(
    f"Roles with 1 record       : {(role_counts == 1).sum():,}"
)

print(
    f"Roles with <= 5 records   : {(role_counts <= 5).sum():,}"
)

print(
    f"Roles with <= 10 records  : {(role_counts <= 10).sum():,}"
)


# ============================================================
# 13. EXPERIENCE SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("EXPERIENCE SUMMARY")
print("=" * 60)

print(
    df["experience_years"]
    .describe()
)


# ============================================================
# 14. SALARY SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("SALARY SUMMARY")
print("=" * 60)

print(
    df["salary_target"]
    .describe()
)


# ============================================================
# 15. SALARY BY DOMAIN
# ============================================================

print("\n" + "=" * 60)
print("SALARY BY DOMAIN")
print("=" * 60)

salary_by_domain = (
    df.groupby("domain")["salary_target"]
    .agg(
        records="count",
        average_salary="mean",
        median_salary="median",
        minimum_salary="min",
        maximum_salary="max"
    )
    .sort_values(
        "records",
        ascending=False
    )
)

salary_by_domain[
    [
        "records",
        "average_salary",
        "median_salary",
        "minimum_salary",
        "maximum_salary"
    ]
] = salary_by_domain[
    [
        "records",
        "average_salary",
        "median_salary",
        "minimum_salary",
        "maximum_salary"
    ]
].round(2)

print(
    salary_by_domain
)


# ============================================================
# 16. SALARY BY TOP ROLES
# ============================================================

print("\n" + "=" * 60)
print("SALARY BY TOP ROLES")
print("=" * 60)

top_roles = (
    df["role"]
    .value_counts()
    .head(20)
    .index
)

salary_by_role = (
    df[
        df["role"].isin(top_roles)
    ]
    .groupby("role")["salary_target"]
    .agg(
        records="count",
        average_salary="mean",
        median_salary="median",
        minimum_salary="min",
        maximum_salary="max"
    )
    .sort_values(
        "records",
        ascending=False
    )
)

salary_by_role = (
    salary_by_role
    .round(2)
)

print(
    salary_by_role
)


# ============================================================
# 17. SKILL COUNT
# ============================================================

def count_skills(skill_text):
    """
    Count the number of skills in a comma-separated list.
    """

    if not skill_text:
        return 0

    skills = [
        skill.strip()
        for skill in str(skill_text).split(",")
        if skill.strip()
    ]

    return len(set(skills))


df["skill_count"] = (
    df["required_skills"]
    .apply(count_skills)
)


# ============================================================
# 18. SKILL COUNT SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("SKILL COUNT SUMMARY")
print("=" * 60)

print(
    df["skill_count"]
    .describe()
)


# ============================================================
# 19. CHECK EMPTY SKILL RECORDS
# ============================================================

empty_skills = (
    df["required_skills"]
    .str.strip()
    .eq("")
)

print("\n" + "=" * 60)
print("EMPTY SKILL RECORDS")
print("=" * 60)

print(
    f"Records without skills: {empty_skills.sum():,}"
)


# ============================================================
# 20. REMOVE RECORDS WITHOUT USEFUL FEATURES
# ============================================================

before_rows = len(df)

df = df[
    df["domain"].ne("")
    & df["role"].ne("")
    & df["salary_target"].notna()
].copy()

after_rows = len(df)

print("\n" + "=" * 60)
print("FEATURE VALIDATION")
print("=" * 60)

print(
    f"Rows before validation : {before_rows:,}"
)

print(
    f"Rows after validation  : {after_rows:,}"
)

print(
    f"Rows removed           : {before_rows - after_rows:,}"
)


# ============================================================
# 21. SELECT ML-RELEVANT COLUMNS
# ============================================================

ml_columns = [
    "title",
    "required_skills",
    "experience_years",
    "domain",
    "role",
    "skill_count",
    "salary_target"
]

ml_df = (
    df[ml_columns]
    .copy()
)


# ============================================================
# 22. FINAL ML DATASET CHECK
# ============================================================

print("\n" + "=" * 60)
print("FINAL ML DATASET")
print("=" * 60)

print(
    f"Rows    : {len(ml_df):,}"
)

print(
    f"Columns : {len(ml_df.columns)}"
)

print("\nColumns:")

for column in ml_df.columns:
    print(f"✓ {column}")


# ============================================================
# 23. FINAL MISSING VALUE CHECK
# ============================================================

print("\n" + "=" * 60)
print("FINAL MISSING VALUES")
print("=" * 60)

print(
    ml_df.isnull().sum()
)


# ============================================================
# 24. SAMPLE DATA
# ============================================================

print("\n" + "=" * 60)
print("SAMPLE ML DATA")
print("=" * 60)

print(
    ml_df.head(10).to_string(
        index=False
    )
)


# ============================================================
# 25. SAVE ML-READY DATASET
# ============================================================

os.makedirs(
    os.path.dirname(OUTPUT_DATASET_PATH),
    exist_ok=True
)

ml_df.to_excel(
    OUTPUT_DATASET_PATH,
    index=False
)


# ============================================================
# 26. COMPLETION
# ============================================================

print("\n" + "=" * 60)
print("STEP 4 COMPLETED")
print("=" * 60)

print(
    "ML-ready dataset saved to:"
)

print(
    OUTPUT_DATASET_PATH
)

FEATURE DATASET LOADED
Rows    : 32,505
Columns : 25

DATA TYPES
title               object
required_skills     object
experience_years     int64
domain              object
role                object
salary_target        int64
dtype: object

MISSING VALUES
title               0
required_skills     0
experience_years    0
domain              0
role                0
salary_target       0
dtype: int64

DOMAIN DISTRIBUTION
domain
Other                           14448
AI / Machine Learning           14072
Cloud / DevOps                    874
Web Development                   674
Software Development              666
Data Engineering                  533
QA / Testing                      374
Data Science                      333
Product / Business Analytics      208
Cybersecurity                     149
UI / UX                            98
Mobile Development                 76
Name: count, dtype: int64

DOMAIN PERCENTAGE
domain
Other                           44.45
AI / Machine Learning   

In [6]:
# ============================================================
# CareerIQ
# Salary Prediction - Step 5
# Model Training & Comparison
# ============================================================

import os
import joblib
import numpy as np
import pandas as pd
import sys
print(sys.executable)

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from xgboost import XGBRegressor


# ============================================================
# 1. PATHS
# ============================================================

ML_DATASET_PATH = (
    "../data/salary_prediction/processed/"
    "salary_prediction_ml_ready.xlsx"
)

MODEL_DIR = "../models/salary_prediction"

BEST_MODEL_PATH = (
    "../models/salary_prediction/"
    "salary_prediction_model.pkl"
)

PREPROCESSOR_PATH = (
    "../models/salary_prediction/"
    "salary_prediction_preprocessor.pkl"
)

RESULTS_PATH = (
    "../data/salary_prediction/processed/"
    "model_comparison.xlsx"
)


# ============================================================
# 2. LOAD DATASET
# ============================================================

df = pd.read_excel(
    ML_DATASET_PATH
)

print("=" * 60)
print("ML DATASET LOADED")
print("=" * 60)

print(f"Rows    : {len(df):,}")
print(f"Columns : {len(df.columns)}")


# ============================================================
# 3. SELECT FEATURES
# ============================================================

feature_columns = [
    "required_skills",
    "experience_years",
    "domain",
    "role",
    "skill_count"
]

target_column = "salary_target"


# ============================================================
# 4. CHECK REQUIRED COLUMNS
# ============================================================

required_columns = (
    feature_columns +
    [target_column]
)

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns: {missing_columns}"
    )


# ============================================================
# 5. PREPARE DATA
# ============================================================

model_df = (
    df[
        required_columns
    ]
    .copy()
)

model_df["required_skills"] = (
    model_df["required_skills"]
    .fillna("")
    .astype(str)
)

model_df["domain"] = (
    model_df["domain"]
    .fillna("Other")
    .astype(str)
)

model_df["role"] = (
    model_df["role"]
    .fillna("Other")
    .astype(str)
)

model_df["experience_years"] = (
    pd.to_numeric(
        model_df["experience_years"],
        errors="coerce"
    )
    .fillna(0)
)

model_df["skill_count"] = (
    pd.to_numeric(
        model_df["skill_count"],
        errors="coerce"
    )
    .fillna(0)
)

model_df[target_column] = (
    pd.to_numeric(
        model_df[target_column],
        errors="coerce"
    )
)

model_df = (
    model_df
    .dropna(
        subset=[target_column]
    )
    .copy()
)


# ============================================================
# 6. X AND y
# ============================================================

X = model_df[
    feature_columns
]

y = model_df[
    target_column
]

print("\n" + "=" * 60)
print("MODEL DATA")
print("=" * 60)

print(f"Features : {X.shape}")
print(f"Target   : {y.shape}")


# ============================================================
# 7. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42
    )
)

print("\n" + "=" * 60)
print("TRAIN / TEST SPLIT")
print("=" * 60)

print(
    f"Training records : {len(X_train):,}"
)

print(
    f"Testing records  : {len(X_test):,}"
)


# ============================================================
# 8. PREPROCESSOR
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[

        (
            "skills",
            TfidfVectorizer(
                max_features=1500,
                ngram_range=(1, 2),
                min_df=2
            ),
            "required_skills"
        ),

        (
            "domain",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            ["domain"]
        ),

        (
            "role",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            ["role"]
        ),

        (
            "numeric",
            "passthrough",
            [
                "experience_years",
                "skill_count"
            ]
        )
    ],
    remainder="drop"
)


# ============================================================
# 9. MODELS
# ============================================================

models = {

    "Random Forest": RandomForestRegressor(
        n_estimators=250,
        max_depth=20,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=250,
        learning_rate=0.05,
        max_depth=5,
        min_samples_leaf=3,
        random_state=42
    ),

    "XGBoost": XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        min_child_weight=3,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )
}


# ============================================================
# 10. TRAIN AND EVALUATE
# ============================================================

results = []

trained_models = {}

print("\n" + "=" * 60)
print("MODEL TRAINING")
print("=" * 60)


for model_name, model in models.items():

    print(
        f"\nTraining {model_name}..."
    )

    pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor
            ),
            (
                "model",
                model
            )
        ]
    )

    # Train
    pipeline.fit(
        X_train,
        y_train
    )

    # Predict
    predictions = pipeline.predict(
        X_test
    )

    # Metrics
    mae = mean_absolute_error(
        y_test,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            predictions
        )
    )

    r2 = r2_score(
        y_test,
        predictions
    )

    results.append(
        {
            "model": model_name,
            "mae": mae,
            "rmse": rmse,
            "r2": r2
        }
    )

    trained_models[
        model_name
    ] = pipeline

    print(
        f"MAE  : ₹{mae:,.2f}"
    )

    print(
        f"RMSE : ₹{rmse:,.2f}"
    )

    print(
        f"R²   : {r2:.4f}"
    )


# ============================================================
# 11. MODEL COMPARISON
# ============================================================

results_df = pd.DataFrame(
    results
)

results_df = (
    results_df
    .sort_values(
        by="mae",
        ascending=True
    )
    .reset_index(
        drop=True
    )
)


print("\n" + "=" * 60)
print("MODEL COMPARISON")
print("=" * 60)

print(
    results_df.to_string(
        index=False
    )
)


# ============================================================
# 12. SELECT BEST MODEL
# ============================================================

best_model_name = (
    results_df
    .iloc[0]["model"]
)

best_model = (
    trained_models[
        best_model_name
    ]
)


print("\n" + "=" * 60)
print("BEST MODEL")
print("=" * 60)

print(
    f"Selected model: {best_model_name}"
)


# ============================================================
# 13. BEST MODEL METRICS
# ============================================================

best_metrics = (
    results_df[
        results_df["model"]
        == best_model_name
    ]
    .iloc[0]
)

print(
    f"\nMAE  : ₹{best_metrics['mae']:,.2f}"
)

print(
    f"RMSE : ₹{best_metrics['rmse']:,.2f}"
)

print(
    f"R²   : {best_metrics['r2']:.4f}"
)


# ============================================================
# 14. CREATE MODEL DIRECTORY
# ============================================================

os.makedirs(
    MODEL_DIR,
    exist_ok=True
)


# ============================================================
# 15. SAVE BEST MODEL
# ============================================================

joblib.dump(
    best_model,
    BEST_MODEL_PATH
)


# ============================================================
# 16. SAVE MODEL COMPARISON
# ============================================================

results_df.to_excel(
    RESULTS_PATH,
    index=False
)


# ============================================================
# 17. SAVE MODEL INFORMATION
# ============================================================

model_info = {
    "model_name": best_model_name,
    "features": feature_columns,
    "target": target_column,
    "mae": float(
        best_metrics["mae"]
    ),
    "rmse": float(
        best_metrics["rmse"]
    ),
    "r2": float(
        best_metrics["r2"]
    )
}

MODEL_INFO_PATH = (
    "../models/salary_prediction/"
    "model_info.pkl"
)

joblib.dump(
    model_info,
    MODEL_INFO_PATH
)


# ============================================================
# 18. FINAL OUTPUT
# ============================================================

print("\n" + "=" * 60)
print("STEP 5 COMPLETED")
print("=" * 60)

print(
    f"Best model : {best_model_name}"
)

print(
    f"Model saved to:\n"
    f"{BEST_MODEL_PATH}"
)

print(
    f"\nComparison saved to:\n"
    f"{RESULTS_PATH}"
)

print(
    f"\nModel information saved to:\n"
    f"{MODEL_INFO_PATH}"
)

c:\anaconda3\python.exe
ML DATASET LOADED
Rows    : 32,505
Columns : 7

MODEL DATA
Features : (32505, 5)
Target   : (32505,)

TRAIN / TEST SPLIT
Training records : 26,004
Testing records  : 6,501

MODEL TRAINING

Training Random Forest...
MAE  : ₹257,136.32
RMSE : ₹452,238.31
R²   : 0.6072

Training Gradient Boosting...
MAE  : ₹254,518.07
RMSE : ₹439,730.93
R²   : 0.6286

Training XGBoost...
MAE  : ₹248,269.05
RMSE : ₹432,900.16
R²   : 0.6401

MODEL COMPARISON
            model           mae          rmse       r2
          XGBoost 248269.046875 432900.163197 0.640064
Gradient Boosting 254518.073155 439730.931196 0.628616
    Random Forest 257136.315949 452238.309049 0.607189

BEST MODEL
Selected model: XGBoost

MAE  : ₹248,269.05
RMSE : ₹432,900.16
R²   : 0.6401

STEP 5 COMPLETED
Best model : XGBoost
Model saved to:
../models/salary_prediction/salary_prediction_model.pkl

Comparison saved to:
../data/salary_prediction/processed/model_comparison.xlsx

Model information saved to:
../mod

In [10]:
import pandas as pd

results = pd.read_excel(
    "../data/salary_prediction/processed/model_comparison.xlsx"
)

print(results.to_string(index=False))

            model           mae          rmse       r2
          XGBoost 248269.046875 432900.163197 0.640064
Gradient Boosting 254518.073155 439730.931196 0.628616
    Random Forest 257136.315949 452238.309049 0.607189


In [15]:
# ============================================================
# STEP 5.5 — IMPROVED SALARY PREDICTION MODEL
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from xgboost import XGBRegressor


# ============================================================
# 1. LOAD DATA
# ============================================================

input_path = (
    "../data/salary_prediction/processed/"
    "salary_prediction_features.xlsx"
)

df = pd.read_excel(input_path)

print("Original dataset shape:", df.shape)


# ============================================================
# 2. CHECK REQUIRED COLUMNS
# ============================================================

required_columns = [
    "title_clean",
    "skills_clean",
    "description_clean",
    "location",
    "experience_years",
    "domain",
    "role",
    "required_skills",
    "salary_target"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns: {missing_columns}"
    )


print("All required columns found.")


# ============================================================
# 3. CLEAN TEXT COLUMNS
# ============================================================

text_columns = [
    "title_clean",
    "skills_clean",
    "description_clean",
    "location",
    "domain",
    "role",
    "required_skills"
]

for col in text_columns:

    df[col] = (
        df[col]
        .fillna("")
        .astype(str)
        .str.strip()
    )


# ============================================================
# 4. CLEAN EXPERIENCE
# ============================================================

df["experience_years"] = pd.to_numeric(
    df["experience_years"],
    errors="coerce"
)

df["experience_years"] = (
    df["experience_years"]
    .fillna(0)
)


# ============================================================
# 5. CLEAN SALARY
# ============================================================

df["salary_target"] = pd.to_numeric(
    df["salary_target"],
    errors="coerce"
)

df = df[
    df["salary_target"].notna()
].copy()

df = df[
    df["salary_target"] > 0
].copy()


# ============================================================
# 6. REMOVE EXTREME SALARY VALUES
# ============================================================

df = df[
    (df["salary_target"] >= 78000)
    &
    (df["salary_target"] <= 4500000)
].copy()


print(
    "Rows after salary cleaning:",
    len(df)
)


# ============================================================
# 7. CREATE SKILL COUNT
# ============================================================

def count_skills(value):

    if not value:
        return 0

    text = str(value)

    for separator in [
        ",",
        ";",
        "|",
        "\n"
    ]:
        text = text.replace(
            separator,
            ","
        )

    skills = [
        skill.strip().lower()
        for skill in text.split(",")
        if skill.strip()
    ]

    return len(set(skills))


df["skill_count"] = (
    df["required_skills"]
    .apply(count_skills)
)


# ============================================================
# 8. CREATE COMBINED TEXT FEATURES
# ============================================================

df["skills_text"] = (
    df["skills_clean"]
    + " "
    + df["required_skills"]
)

df["job_text"] = (
    df["title_clean"]
    + " "
    + df["skills_clean"]
    + " "
    + df["description_clean"]
)


# ============================================================
# 9. DISPLAY DATASET INFORMATION
# ============================================================

print("\nFinal dataset shape:", df.shape)

print("\nDomain distribution:")
print(
    df["domain"]
    .value_counts()
    .head(15)
)


# ============================================================
# 10. FEATURES
# ============================================================

X = df[
    [
        "skills_text",
        "job_text",
        "location",
        "experience_years",
        "skill_count"
    ]
]

y = df["salary_target"]


print("\nFeature columns:")
print(X.columns.tolist())

print("\nTarget:")
print("salary_target")


# ============================================================
# 11. LOG TRANSFORM TARGET
# ============================================================

y_log = np.log1p(y)


# ============================================================
# 12. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_log,
    test_size=0.20,
    random_state=42
)


print("\nTraining rows:", len(X_train))
print("Testing rows:", len(X_test))


# ============================================================
# 13. PREPROCESSING
# ============================================================

preprocessor = ColumnTransformer(

    transformers=[

        (
            "skills_tfidf",

            TfidfVectorizer(
                max_features=3000,
                ngram_range=(1, 2),
                min_df=2
            ),

            "skills_text"
        ),

        (
            "job_tfidf",

            TfidfVectorizer(
                max_features=5000,
                ngram_range=(1, 2),
                min_df=3
            ),

            "job_text"
        ),

        (
            "location",

            OneHotEncoder(
                handle_unknown="ignore"
            ),

            ["location"]
        ),

        (
            "numeric",

            SimpleImputer(
                strategy="median"
            ),

            [
                "experience_years",
                "skill_count"
            ]
        )
    ],

    remainder="drop"
)


# ============================================================
# 14. IMPROVED XGBOOST
# ============================================================

model = XGBRegressor(

    n_estimators=700,

    learning_rate=0.035,

    max_depth=7,

    min_child_weight=3,

    subsample=0.85,

    colsample_bytree=0.85,

    reg_alpha=0.1,

    reg_lambda=1.5,

    objective="reg:squarederror",

    random_state=42,

    n_jobs=-1
)


# ============================================================
# 15. COMPLETE PIPELINE
# ============================================================

pipeline = Pipeline(

    steps=[

        (
            "preprocessor",
            preprocessor
        ),

        (
            "model",
            model
        )
    ]
)


# ============================================================
# 16. TRAIN
# ============================================================

print("\n========================================")
print("TRAINING IMPROVED XGBOOST")
print("========================================")

pipeline.fit(
    X_train,
    y_train
)

print("Training completed.")


# ============================================================
# 17. PREDICTION
# ============================================================

pred_log = pipeline.predict(
    X_test
)


# Convert log salary back to INR

pred_salary = np.expm1(
    pred_log
)

actual_salary = np.expm1(
    y_test
)


# ============================================================
# 18. METRICS
# ============================================================

mae = mean_absolute_error(
    actual_salary,
    pred_salary
)

rmse = np.sqrt(
    mean_squared_error(
        actual_salary,
        pred_salary
    )
)

r2 = r2_score(
    actual_salary,
    pred_salary
)


# ============================================================
# 19. DISPLAY RESULTS
# ============================================================

print("\n========================================")
print("IMPROVED XGBOOST RESULTS")
print("========================================")

print(
    f"MAE  : ₹{mae:,.2f}"
)

print(
    f"RMSE : ₹{rmse:,.2f}"
)

print(
    f"R²   : {r2:.4f}"
)


# ============================================================
# 20. SAMPLE PREDICTIONS
# ============================================================

prediction_results = pd.DataFrame({

    "Actual Salary":
        actual_salary.values,

    "Predicted Salary":
        pred_salary

})

prediction_results["Difference"] = (
    prediction_results["Predicted Salary"]
    -
    prediction_results["Actual Salary"]
)


print("\n========================================")
print("SAMPLE PREDICTIONS")
print("========================================")

print(
    prediction_results
    .head(10)
    .to_string(index=False)
)

Original dataset shape: (32505, 25)
All required columns found.
Rows after salary cleaning: 32505

Final dataset shape: (32505, 28)

Domain distribution:
domain
Other                           14448
AI / Machine Learning           14072
Cloud / DevOps                    874
Web Development                   674
Software Development              666
Data Engineering                  533
QA / Testing                      374
Data Science                      333
Product / Business Analytics      208
Cybersecurity                     149
UI / UX                            98
Mobile Development                 76
Name: count, dtype: int64

Feature columns:
['skills_text', 'job_text', 'location', 'experience_years', 'skill_count']

Target:
salary_target

Training rows: 26004
Testing rows: 6501

TRAINING IMPROVED XGBOOST
Training completed.

IMPROVED XGBOOST RESULTS
MAE  : ₹218,260.94
RMSE : ₹422,417.32
R²   : 0.6573

SAMPLE PREDICTIONS
 Actual Salary  Predicted Salary    Difference
      32

In [18]:
print("========================================")
print("IMPROVED XGBOOST RESULTS")
print("========================================")

mae = mean_absolute_error(
    np.expm1(y_test),
    pred_salary
)

rmse = np.sqrt(
    mean_squared_error(
        np.expm1(y_test),
        pred_salary
    )
)

r2 = r2_score(
    np.expm1(y_test),
    pred_salary
)

print(f"MAE  : ₹{mae:,.2f}")
print(f"RMSE : ₹{rmse:,.2f}")
print(f"R²   : {r2:.4f}")

IMPROVED XGBOOST RESULTS
MAE  : ₹218,260.94
RMSE : ₹422,417.32
R²   : 0.6573


In [21]:
# ============================================================
# STEP 6 — SAVE FINAL SALARY PREDICTION MODEL
# ============================================================

import os
import pickle


# ------------------------------------------------------------
# 1. CREATE MODEL DIRECTORY
# ------------------------------------------------------------

model_dir = "../models/salary_prediction"

os.makedirs(
    model_dir,
    exist_ok=True
)


# ------------------------------------------------------------
# 2. SAVE TRAINED PIPELINE
# ------------------------------------------------------------

model_path = (
    f"{model_dir}/salary_prediction_model.pkl"
)

with open(model_path, "wb") as file:

    pickle.dump(
        pipeline,
        file
    )


print("Model saved successfully:")
print(model_path)


# ------------------------------------------------------------
# 3. SAVE MODEL INFORMATION
# ------------------------------------------------------------

model_info = {

    "model": "Improved XGBoost",

    "target": "salary_target",

    "currency": "INR",

    "mae": float(mae),

    "rmse": float(rmse),

    "r2": float(r2),

    "training_rows": int(len(X_train)),

    "testing_rows": int(len(X_test)),

    "features": [
        "skills_text",
        "job_text",
        "location",
        "experience_years",
        "skill_count"
    ],

    "n_estimators": 700,

    "learning_rate": 0.035,

    "max_depth": 7,

    "random_state": 42
}


info_path = (
    f"{model_dir}/model_info.pkl"
)

with open(info_path, "wb") as file:

    pickle.dump(
        model_info,
        file
    )


print("\nModel information saved successfully:")
print(info_path)


# ------------------------------------------------------------
# 4. VERIFY FILES
# ------------------------------------------------------------

print("\n========================================")
print("SALARY MODEL SAVING COMPLETE")
print("========================================")

print(
    f"MAE  : ₹{mae:,.2f}"
)

print(
    f"RMSE : ₹{rmse:,.2f}"
)

print(
    f"R²   : {r2:.4f}"
)

print("\nFiles created:")

print(
    "1.",
    model_path
)

print(
    "2.",
    info_path
)

Model saved successfully:
../models/salary_prediction/salary_prediction_model.pkl

Model information saved successfully:
../models/salary_prediction/model_info.pkl

SALARY MODEL SAVING COMPLETE
MAE  : ₹218,260.94
RMSE : ₹422,417.32
R²   : 0.6573

Files created:
1. ../models/salary_prediction/salary_prediction_model.pkl
2. ../models/salary_prediction/model_info.pkl
